In [34]:
# 1) ติดตั้ง (ครั้งแรกเท่านั้น)
!pip install -q pandas scikit-learn joblib openpyxl

In [35]:
# 2) โหลดชิต "Sheet2"
import pandas as pd
df = pd.read_excel("/content/sample_data/data.xlsx", sheet_name="Sheet2")

print(df)

       order_id shop_guid  app_source              order_time  distance_km  \
0     ORD300000    SHOP_C    LINE_MAN 2025-04-27 12:12:29.222         4.21   
1     ORD300001    SHOP_I   Robinhood 2025-04-27 10:48:20.033         7.51   
2     ORD300002    SHOP_D   Robinhood 2025-04-27 10:33:29.005         3.51   
3     ORD300003    SHOP_D   Robinhood 2025-04-27 19:58:15.588         7.50   
4     ORD300004    SHOP_A    GrabFood 2025-04-27 17:30:57.995         6.72   
...         ...       ...         ...                     ...          ...   
9995  ORD309995    SHOP_C  ShopeeFood 2025-04-27 18:50:29.551         6.87   
9996  ORD309996    SHOP_F    GrabFood 2025-04-27 11:51:32.788         1.92   
9997  ORD309997    SHOP_A    LINE_MAN 2025-04-27 19:59:48.538         7.26   
9998  ORD309998    SHOP_D   Robinhood 2025-04-27 14:34:29.341         5.04   
9999  ORD309999    SHOP_D  ShopeeFood 2025-04-27 13:23:22.600         2.46   

      queue_size  prep_time_sec  traffic_sec  rain_flag  eta_ac

In [36]:
# 3) ฟีเจอร์กับเป้าหมาย
X = df[['distance_km','queue_size','prep_time_sec',
        'traffic_sec','rain_flag']]
y = df['eta_actual_min']

print(X)
print(y)

      distance_km  queue_size  prep_time_sec  traffic_sec  rain_flag
0            4.21           7            266          331          0
1            7.51           2            955         1041          0
2            3.51           4           1070         1966          0
3            7.50           9           1064          563          0
4            6.72           5           1063         1839          1
...           ...         ...            ...          ...        ...
9995         6.87           8           1132         1909          0
9996         1.92          10           1158         1064          0
9997         7.26           9            560         1608          0
9998         5.04           3            912          966          0
9999         2.46           3            784         1547          0

[10000 rows x 5 columns]
0       26.7
1       31.4
2       60.5
3       47.4
4       55.2
        ... 
9995    68.1
9996    50.9
9997    55.5
9998    40.3
9999    43.8
Nam

In [37]:
# 4) แบ่ง Train / Test  (ต้องทำก่อน fit!)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

In [38]:
# 5) สร้างและฝึก Linear Regression
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X_train, y_train)

In [40]:
# 6) ประเมินความแม่น — MAE, RMSE, MAPE, P50/P90, Accuracy±5/10
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)
error  = np.abs(y_test - y_pred)

mae  = mean_absolute_error(y_test, y_pred)
# Calculate RMSE using NumPy to avoid the 'squared' argument
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = (error / y_test).mean() * 100
p50  = np.percentile(error, 50)
p90  = np.percentile(error, 90)
acc5  = (error <= 5).mean()*100
acc10 = (error <= 10).mean()*100

print(f"MAE     : {mae:.2f}  min")
print(f"RMSE    : {rmse:.2f}  min")
print(f"MAPE    : {mape:.1f} %")
print(f"P50|P90 : {p50:.2f} | {p90:.2f}  min")
print(f"±5 min  : {acc5:.1f}%  |  ±10 min : {acc10:.1f}%")

MAE     : 2.39  min
RMSE    : 3.01  min
MAPE    : 6.8 %
P50|P90 : 1.98 | 4.95  min
±5 min  : 90.3%  |  ±10 min : 100.0%


In [41]:
# 7) บันทึกโมเดล
import joblib
joblib.dump(model, "eta_model.pkl")

# 8) โหลดกลับและทดสอบใบใหม่
loaded = joblib.load("eta_model.pkl")
test_order = pd.DataFrame([{
    "distance_km": 3.2,
    "queue_size": 4,
    "prep_time_sec": 780,
    "traffic_sec": 900,
    "rain_flag": 1
}])
print(f"\nETA คาด ≈ {loaded.predict(test_order)[0]:.1f} นาที")


ETA คาด ≈ 36.1 นาที


In [42]:
# ---------- 1) บันทึกค่าพยากรณ์ ----------
pred_log = pd.DataFrame([{
    'order_id':    'XT123',
    'order_time':  pd.Timestamp('2025-04-27 12:34:00'),
    'eta_pred_min': loaded.predict(test_order)[0]
}])

# ---------- 2) สมมุติว่าได้รับเวลาส่งจริง ----------
delivered_ts = pd.Timestamp('2025-04-27 13:02:00')   # สมมุติ

# ---------- 3) คำนวณ ETA จริง & Error ----------
pred_log['delivered_ts']   = delivered_ts
pred_log['eta_actual_min'] = (delivered_ts - pred_log['order_time']).dt.total_seconds()/60
pred_log['error_min']      = abs(pred_log['eta_actual_min'] - pred_log['eta_pred_min'])

print(pred_log[['eta_pred_min','eta_actual_min','error_min']])


   eta_pred_min  eta_actual_min  error_min
0     36.089073            28.0   8.089073


# ส่วนใหม่